In [0]:
# шлях до файлів з даними
from pyspark.sql.types import *

mount_name = '/mnt/robot-dreams-source-data-mount/home-work-1/nyc_taxi/'

In [0]:
# функція для зчитування файлів та приведення до одного формату
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import col

def create_dataframe_with_taxi_data(data_url, file_schema):
    taxi_dfs = []

    for file in spark.read.parquet(data_url).inputFiles():
        denormalized_df = spark.read.parquet(file)
        taxi_dfs.append(
                denormalized_df.select(
                [col(field.name).cast(field.dataType).alias(field.name)
                for field in file_schema.fields]
            )
        )

    return reduce(DataFrame.union, taxi_dfs)

In [0]:
# вичитування файлів з даними по зелених таксі із заданою схемою
schema_green_taxi = StructType([
    StructField("VendorID", LongType(), True),
    StructField("lpep_pickup_datetime", TimestampType(), True),
    StructField("lpep_dropoff_datetime", TimestampType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("ehail_fee", IntegerType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("payment_type", LongType(), True),
    StructField("trip_type", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True)
])

green_taxi_data_url = f"{mount_name}green/*"
print(f"Using Green Taxi data from: {green_taxi_data_url}")
green_taxi_df = create_dataframe_with_taxi_data(green_taxi_data_url, schema_green_taxi)
print(f"Green Taxi data loaded. Records: {green_taxi_df.count()}")
green_taxi_df.printSchema()

Using Green Taxi data from: /mnt/robot-dreams-source-data-mount/home-work-1/nyc_taxi/green/2023*
Green Taxi data loaded. Records: 592643
root
 |-- VendorID: long (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: integer (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: double (nullable = true)
 |-- congestion_su

In [0]:
# вичитування файлів з даними по жовтих таксі із заданою схемою

schema_yellow_taxi = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("payment_type", LongType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", IntegerType(), True)
])

yellow_taxi_data_url = f"{mount_name}yellow/*"
print(f"Using Yellow Taxi data from: {yellow_taxi_data_url}")
yellow_taxi_df = create_dataframe_with_taxi_data(yellow_taxi_data_url, schema_yellow_taxi)
print(f"Yellow Taxi data loaded. Records: {yellow_taxi_df.count()}")
yellow_taxi_df.printSchema()

Using Yellow Taxi data from: /mnt/robot-dreams-source-data-mount/home-work-1/nyc_taxi/yellow/2023*
Yellow Taxi data loaded. Records: 28071659
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)


In [0]:
# додаємо колонку з кольором таксі у датафрейми
from pyspark.sql.functions import lit, unix_timestamp

green_taxi_new_column_df = green_taxi_df.withColumn('taxi_type', lit('green'))
yellow_taxi_new_column_df = yellow_taxi_df.withColumn('taxi_type', lit('yellow'))

In [0]:
# перейменовуємо стовпчики в меншому датасеті та об'єднуємо їх

green_taxi_reshaped_df = green_taxi_new_column_df.withColumnsRenamed({'lpep_pickup_datetime': 'tpep_pickup_datetime', 'lpep_dropoff_datetime': 'tpep_dropoff_datetime'})

raw_trips_df = yellow_taxi_new_column_df.unionByName(green_taxi_reshaped_df, allowMissingColumns=True)

In [0]:
# додаємо нові стопвчики з даними  duration_min — тривалість поїзки у хвилинах, pickup_hour — годину початку поїздки, pickup_day_of_week — день тижня
from pyspark.sql.functions import hour, dayofweek

raw_trips_new_columns_df = raw_trips_df.withColumn(
    'duration_min', (unix_timestamp('tpep_dropoff_datetime') - unix_timestamp('tpep_pickup_datetime')) / 60
).withColumn(
    'pickup_hour', hour('tpep_pickup_datetime')
).withColumn(
    'pickup_day_of_week', dayofweek('tpep_pickup_datetime')
)

In [0]:
# вилучаємо поїздки з відстанню < 0.1 км (дані в милях, 0.1 км ≈ 0.062 милі), тарифом < $2, тривалістю < 1 хв.
raw_trips_filtered_df = raw_trips_new_columns_df.filter(col('trip_distance') > 0.062).filter(col('fare_amount') >= 2).filter(col('duration_min') >= 1)

In [0]:
# читаємо дані про локації з csv файла
schema_csv = StructType([
    StructField("LocationID", LongType(), True),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True)
])
df_csv = spark.read.option("header", True).schema(schema_csv).csv(f"{mount_name}taxi_zone_lookup.csv")

In [0]:
from pyspark.sql.functions import broadcast

raw_trips_final_df = raw_trips_filtered_df.join(
    broadcast(df_csv), raw_trips_filtered_df.PULocationID == df_csv.LocationID, how="left_outer"
).drop('LocationID', 'Borough', 'service_zone').withColumnRenamed('Zone', 'pickup_zone')

raw_trips_final_df = raw_trips_final_df.join(
    broadcast(df_csv), raw_trips_final_df.DOLocationID == df_csv.LocationID, how="left_outer"
).drop('LocationID', 'Borough', 'service_zone').withColumnRenamed('Zone', 'dropoff_zone')

In [0]:
%sql
create catalog if not exists szhylina_nyc_catalog
managed location 's3://szhylina-hw/homework-7';

In [0]:
%sql
use catalog szhylina_nyc_catalog;
create schema if not exists trips_schema;

In [0]:
%sql
use schema trips_schema;
create table if not exists raw_trips (
  VendorID long,
  tpep_pickup_datetime timestamp,
  tpep_dropoff_datetime timestamp,
  store_and_fwd_flag string,
  RatecodeID long,
  PULocationID long,
  DOLocationID long,
  passenger_count long,
  trip_distance double,
  fare_amount double,
  extra double,
  mta_tax double,
  tip_amount double,
  tolls_amount double,
  improvement_surcharge double,
  total_amount double,
  payment_type long,
  congestion_surcharge double,
  airport_fee integer,
  taxi_type string,
  ehail_fee integer,
  trip_type double,
  duration_min double,
  pickup_hour integer,
  pickup_day_of_week integer,
  pickup_zone string,
  dropoff_zone string
);

In [0]:
raw_trips_final_df.show(5)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------------------+------------+------------+--------------------+-----------+---------+---------+---------+------------------+-----------+------------------+-----------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|payment_type|congestion_surcharge|airport_fee|taxi_type|ehail_fee|trip_type|      duration_min|pickup_hour|pickup_day_of_week|      pickup_zone|        dropoff_zone|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------------------+---------

In [0]:
# запис даних в unity catalog
raw_trips_final_df.repartition(50).write.format("delta").mode("overwrite").saveAsTable("szhylina_nyc_catalog.trips_schema.raw_trips")

In [0]:
# формуємо датафрейм zone_summary з агрегованими даними по pickup_zone
from pyspark.sql.functions import count, avg, max, min, sum, when

zone_summary = raw_trips_final_df.groupBy("pickup_zone").agg(
    count("*").alias("total_trips"),
    avg("trip_distance").alias("avg_trip_distance"),
    avg("total_amount").alias("avg_total_amount"),
    avg("tip_amount").alias("avg_tip_amount"),
    max("trip_distance").alias("max_trip_distance"),
    min("tip_amount").alias("min_tip_amount"),
    sum(when(col("taxi_type") == "raw_trips_final_df", 1).otherwise(0)).alias("yellow_count"),
    sum(when(col("taxi_type") == "green", 1).otherwise(0)).alias("green_count")
)

zone_summary = zone_summary.withColumn("yellow_share", col("yellow_count") / col("total_trips"))
zone_summary = zone_summary.withColumn("green_share", col("green_count") / col("total_trips"))

zone_summary = zone_summary.drop("yellow_count", "green_count")

zone_summary.show(5)

+----------------+-----------+------------------+------------------+------------------+-----------------+--------------+------------+--------------------+
|     pickup_zone|total_trips| avg_trip_distance|  avg_total_amount|    avg_tip_amount|max_trip_distance|min_tip_amount|yellow_share|         green_share|
+----------------+-----------+------------------+------------------+------------------+-----------------+--------------+------------+--------------------+
|       Homecrest|        666|15.856831831831832|  38.6360960960961| 0.962882882882883|          5517.87|           0.0|         0.0| 0.12612612612612611|
|          Corona|       2001| 6.358770614692654| 48.19637181409293|3.8724137931034477|            56.33|           0.0|         0.0|  0.2098950524737631|
|Bensonhurst West|        635| 8.752346456692914| 43.46340157480315| 1.099181102362205|            37.73|           0.0|         0.0| 0.06771653543307087|
|     Westerleigh|         19|24.475263157894737|130.02947368421056| 8

In [0]:
# запис даних в unity catalog
zone_summary.write.format("delta").mode("overwrite").saveAsTable("szhylina_nyc_catalog.trips_schema.zone_summary")

In [0]:
# формуємо датафрейм zone_days_summary з агрегованими даними по днях тижня та зонах

raw_trips_delta_df = spark.table("szhylina_nyc_catalog.trips_schema.raw_trips")

zone_days_summary = raw_trips_delta_df.groupBy("pickup_zone", "pickup_day_of_week").agg(
    count("*").alias("total_trips"),
    avg("duration_min").alias("avg_duration"),
    (sum(when(col("fare_amount") > 30, 1).otherwise(0)) / count("*")).alias("high_fare_share")
)

zone_days_summary.show(5)

+--------------------+------------------+-----------+------------------+-------------------+
|         pickup_zone|pickup_day_of_week|total_trips|      avg_duration|    high_fare_share|
+--------------------+------------------+-----------+------------------+-------------------+
|         Old Astoria|                 7|        978|21.982958418541237|0.18404907975460122|
|Washington Height...|                 6|        902|  21.9789541759054| 0.4212860310421286|
|Marine Park/Floyd...|                 6|         38| 18.81491228070175|0.47368421052631576|
|         Old Astoria|                 4|        954|20.223602375960862|0.15513626834381553|
|            Canarsie|                 6|        451|47.957390983000735| 0.6008869179600886|
+--------------------+------------------+-----------+------------------+-------------------+
only showing top 5 rows


In [0]:
# запис даних в unity catalog
zone_summary.write.format("delta").mode("overwrite").saveAsTable("szhylina_nyc_catalog.trips_schema.zone_days_summary")